<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 30 · Credit Documents, Covenants, and Research Notes

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch30_credit_documents_covenants_research_notes.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This chapter brings GenAI controls to credit documents, where exact legal language carries the risk. You will learn grounded extraction: every covenant field must carry its value, unit, source snippet, and review status before it supports a conclusion. You will compute covenant headroom with the metric direction applied, threshold minus observed value for a maximum leverage covenant and observed value minus threshold for a minimum coverage covenant, and flag thin headroom for analyst review. You will keep extracted facts separate from credit judgment in research notes, and distinguish the synthetic credit memos from the official SEC EDGAR filing metadata stored in the companion data.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script reads the stored covenant extraction table for three issuers and computes headroom with the metric direction applied: threshold minus value for the maximum net-leverage covenant, and value minus threshold for the minimum interest-coverage covenant. It prints the headroom table and flags issuers with headroom below 0.40 for review. Beacon Health's 0.30x coverage headroom is the row that routes to the analyst.

In [ ]:
SCRIPT = ROOT / "code" / "ch30_credit_documents_covenants_research_notes.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Verify the direction logic in the output: Atlas Components and Cedar Retail use threshold minus value on the maximum leverage covenant, while Beacon Health uses value minus threshold on the minimum coverage covenant.
Check the three headroom values against the table: 0.55, 0.30, and 0.55, all in x.
Confirm that only Beacon Health carries the review flag, because 0.30 falls below the 0.40 cutoff; this is thin headroom, not a breach, and all three statuses are pass.
Read the source snippet behind each threshold, such as caps net leverage at 4.75x, as the audit trail for the extracted value.
Keep the synthetic credit memos separate from the real SEC EDGAR filing metadata snapshot in the data folder.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch30_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Covenant threshold check with metric direction

This lab isolates one mechanism: checking extracted covenant values against a small terms table with the direction applied per metric. For a maximum leverage covenant, headroom is the threshold minus the value; for a minimum coverage covenant, it is the value minus the threshold. Negative headroom is a breach, and the terms are illustrative.

In [ ]:
import pandas as pd

terms = pd.DataFrame({  # illustrative covenant terms
    "issuer": ["Atlas", "Beacon", "Cedar"],
    "metric": ["net_leverage", "interest_coverage", "net_leverage"],
    "value": [4.2, 2.1, 4.9],  # extracted ratio, in x
    "threshold": [4.75, 1.80, 4.25],
    "direction": ["max", "min", "max"],  # max: lower is better
})
headroom = []
for row in terms.itertuples():
    if row.direction == "max":  # threshold minus value
        headroom.append(row.threshold - row.value)
    else:  # coverage: value minus threshold
        headroom.append(row.value - row.threshold)
terms["headroom"] = headroom
terms["breach"] = terms.headroom.lt(0)  # negative is a breach
terms

### What to observe

Expected output: Atlas headroom 0.55 (4.75 minus 4.2), Beacon 0.30 (2.1 minus 1.80), and Cedar -0.65 (4.25 minus 4.9), all in x. Cedar's negative headroom breaches the maximum leverage covenant, while Beacon's 0.30 is thin and would warrant analyst review even without a breach.

## Exercises

1. **Direction check.** Recompute headroom for all three issuers without the metric-direction branch and report which row changes sign and meaning. *Hint:* Beacon Health's coverage covenant is the row where a plain difference reverses the interpretation.
2. **Review threshold.** Raise the review cutoff from 0.40 to 0.60 and record which issuers now require review. *Hint:* Both leverage rows sit at 0.55, so they enter the queue together with Beacon Health.
3. **Snippet audit.** For each extraction in ch30_covenant_extractions.csv, read the source snippet and confirm which threshold and metric it supports. *Hint:* The snippet caps net leverage at 4.75x is the audit trail for Atlas Components, not decoration.
4. **Breach simulation.** Add a fourth issuer to the mini-lab terms table whose leverage exceeds its threshold and confirm the breach flag fires. *Hint:* Negative headroom on a max covenant means the observed value is already above the cap.
5. **Facts versus judgment.** Draft a two-sentence research note for Beacon Health that separates the extracted coverage facts from the analyst's interpretation. *Hint:* State the covenant, the snippet, and the headroom; do not turn thin headroom into a default prediction.

For more detail, see Chapter 30 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">